# 05 · News stress-test — 10 live rounds, logged for analysis

A **News-only** harness: plays the News competition (`competition_id=5`) **10 times back to back**,
each round its own logged run (`news_r01`..`news_r10`), then aggregates everything — per-round
accuracy + reached level, the retrieval **source mix** (Guardian / headless-Chromium / gnews), and
**every wrong question WITH the retrieved evidence text** — so we can tell a *retrieval miss* (the
answer wasn't in the evidence) from a *grounding miss* (it was, but the model ignored it).

> Leaderboard attempts are FREE (only the cumulative best counts), so re-running 10 rounds costs us
> nothing on the board. We still pause politely between games (the PDF asks: no rapid requests).

## 1 · Setup — clone/sync the repo, paths, the provided client

In [17]:
# Auto-reload edited src modules on every cell run -- so after a `git pull` the newest code lands without a
# manual importlib.reload or a restart. (Re-run the cell that USES the code, e.g. code-wire.)
# Colab's IPython ships an autoreload that does `from imp import reload`, and `imp` is GONE in Python 3.12 --
# so a tiny `imp` shim (reload only) we install first, then load the extension. BEST-EFFORT: any failure
# caught, so the cell never stalls (fall back: after a src pull, Runtime > Restart to pick changes up).
try:
    import sys as _sys, types as _types, importlib as _importlib
    if 'imp' not in _sys.modules:
        _imp = _types.ModuleType('imp')
        _imp.reload = _importlib.reload          # the one thing the old autoreload.py wants from `imp`.
        _sys.modules['imp'] = _imp
    _ip = get_ipython()
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
    print('autoreload: ON (src edits hot-reload on cell re-run)')
except Exception as _e:
    print(f'autoreload OFF ({type(_e).__name__}: {_e}) -- after a src pull, Runtime > Restart to pick changes up.')

import os, sys

REPO_URL = 'https://github.com/SleepyEveryD/NLP.git'
REPO_ROOT = '/content/NLP'
BRANCH = 'fix-others'
if not os.path.exists(REPO_ROOT):
  !git clone -b {BRANCH} {REPO_URL} {REPO_ROOT}
else:
  # Already cloned -> HARD-SYNC to the latest pushed branch (fetch + force-reset to origin/{BRANCH}).
  # Tracked files are overwritten to match remote; UNTRACKED run outputs are KEPT (experiments/runs/* is
  # gitignored). NOTE: `git pull` updates the FILES on disk -- it does NOT refresh THIS notebook's cells.
  !cd {REPO_ROOT} && git fetch -q origin && git checkout -q -f -B {BRANCH} origin/{BRANCH}

!cd {REPO_ROOT} && echo "on branch:" $(git rev-parse --abbrev-ref HEAD) "@" $(git --no-pager log -1 --oneline)

SRC = os.path.join(REPO_ROOT, 'src')
API_CLIENT = os.path.join(REPO_ROOT, 'NLP_assignment_api_client')
for p in (SRC, API_CLIENT):
  if p not in sys.path:
    sys.path.insert(0, p)
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)

from millionaire_client import MillionaireClient
print('millionaire_client, imported it is.')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
autoreload: ON (src edits hot-reload on cell re-run)
on branch: fix-others @ e85beb5 Merge branch 'fix-others' of https://github.com/SleepyEveryD/NLP into news
Repo root: /content/NLP
millionaire_client, imported it is.


In [18]:
# The inference stack + the client's `requests`, install we do (light it stays). `-U` kept (Colab a stale
# bitsandbytes preinstalls); pandas/requests PINNED to Colab's versions (bare `-U` breaks google-colab/cudf).
!pip install -q -U 'transformers>=4.45.0' 'accelerate>=0.34.0' 'bitsandbytes>=0.46.1' sentencepiece einops pyyaml 'pandas==2.2.2' matplotlib 'requests==2.32.4'
print('Installed, the dependencies are.')

Installed, the dependencies are.


In [19]:
# Headless Chromium -- the live-NEWS body fetch it powers (configs/live.yaml: news_body_mode "browser").
# The relevance gate now ROUTES off-topic Guardian results here, so the browser matters MORE for News.
# Skip this only if you set retrieval.news_body_mode: "off".
!pip install -q playwright
!playwright install chromium
!playwright install-deps

# ARMED? a REAL launch the surest test is. NOT ready -> News falls back to HEADLINES only (crash-safe).
try:
    from playwright.sync_api import sync_playwright
    with sync_playwright() as _p:
        _b = _p.chromium.launch(headless=True); _b.close()
    print('headless Chromium: READY -- live-News body fetch armed.')
except Exception as _e:
    print(f'headless Chromium NOT ready ({type(_e).__name__}: {_e})')
    print('   -> News will use HEADLINES only. Re-run this cell, or set retrieval.news_body_mode: "off".')

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entr

## 2 · Config — News competition + how many rounds

In [20]:
from config import RunConfig

config = RunConfig.from_yaml(os.path.join(REPO_ROOT, 'configs', 'live.yaml'))

# --- The ONLY knobs this notebook needs ---
NEWS_COMP_ID = 5            # News competition id (see the list printed after login).
NUM_ROUNDS   = 10          # how many live News games to play, back to back.
PAUSE_S      = 8.0         # polite gap between games (PDF: no rapid consecutive requests).

config.game.competition_id = NEWS_COMP_ID
config.game.game_mode = 'text'

# The Guardian Open Platform key -- from a Colab secret (NEVER hardcoded). With it, the Guardian fast body
# path is armed; the relevance gate then keeps it ONLY when on-topic, else routes to the browser.
try:
    from google.colab import userdata as _ud
    config.retrieval.guardian_api_key = _ud.get('guardian_key') or ''
except Exception:
    config.retrieval.guardian_api_key = config.retrieval.guardian_api_key or ''

print('mode:', config.mode, '| competition_id:', config.game.competition_id, '(News)')
print('rounds:', NUM_ROUNDS, '| pause between:', PAUSE_S, 's')
print('aim_seconds:', config.game.aim_seconds, '| model:', config.model.name, '|', config.model.quantization)
print('RAG:', 'ON' if config.retrieval.enabled else 'OFF', '| source:', config.retrieval.source,
      '| news_body_mode:', config.retrieval.news_body_mode, '| fetch_bodies:', config.retrieval.news_fetch_bodies)
print('Guardian API:', 'KEY SET' if config.retrieval.guardian_api_key else 'no key -> News uses browser')

mode: live | competition_id: 5 (News)
rounds: 10 | pause between: 8.0 s
aim_seconds: 25.0 | model: Qwen/Qwen2.5-7B-Instruct | 4bit
RAG: ON | source: routed | news_body_mode: browser | fetch_bodies: 3
Guardian API: KEY SET


## 3 · Load + warm up the model

In [21]:
import time
from inference.engine import TransformersEngine

t0 = time.perf_counter()
if 'engine' not in globals():
      engine = TransformersEngine(model_name=config.model.name,
                                  quantization=config.model.quantization,
                                  dtype=config.model.dtype)
      engine.warmup()
else:
      print('engine 已在显存中,跳过加载。')
print(f'Model loaded in {time.perf_counter() - t0:.1f}s')

t0 = time.perf_counter()
engine.warmup()
print(f'Warmup in {time.perf_counter() - t0:.1f}s')

engine 已在显存中,跳过加载。
Model loaded in 0.0s
Warmup in 0.7s


## 4 · Wire the News pipeline + log in to the game

In [22]:
from classify.classifier import QuestionClassifier
from prompting.builder import PromptBuilder
from agent.pipeline import QAPipeline
from tools import default_tools
from retrieval import build_retriever

# Phase 4 RAG: the routing retriever. For News (post-cutoff) `routed` sends questions to the live web
# (Guardian API + relevance gate -> headless-Chromium on the gnews link); `needs_retrieval` gates it.
retriever = build_retriever(config.retrieval)
print('RAG:', (f'ON  source={config.retrieval.source}  top_k={config.retrieval.top_k}') if retriever else 'OFF')

# Exactly the SHARED pipeline competition 5 uses in notebook 03 -- few_shot + RAG + the calculator no-op.
pipeline = QAPipeline(
    engine=engine,
    prompt_builder=PromptBuilder(strategy=config.prompt_strategy),
    classifier=QuestionClassifier(),
    retriever=retriever,
    tools=default_tools(),
    latency_budget_s=config.latency_budget_s,
)
print('News pipeline wired:', config.prompt_strategy, '+ RAG (routed) + classifier-gated tools')

# --- Log in to the real game ---
from google.colab import userdata
from game.client import GameClient

USERNAME = userdata.get('username')
PASSWORD = userdata.get('password')

game_client = GameClient()
game_client.login(USERNAME, PASSWORD)
print('Logged in as', USERNAME)

# The competitions + ids (safe -- starts no timer). Confirm News is id 5 here.
for c in game_client.list_competitions():
    print('  id=', c.id, '|', c.name, '| max_levels=', getattr(c, 'max_levels', '?'))

RAG: ON  source=routed  top_k=3
News pipeline wired: few_shot_v1 + RAG (routed) + classifier-gated tools
Logged in as runjie dai
  id= 0 | Entertainment | max_levels= 15
  id= 1 | Ancient History and Politics | max_levels= 15
  id= 2 | Science and Nature | max_levels= 15
  id= 3 | Maths | max_levels= 15
  id= 4 | Philosophy and Psychology | max_levels= 15
  id= 5 | News | max_levels= 15


## 5 · ▶ Play 10 live News rounds  (each its own logged run)

Plays the News game `NUM_ROUNDS` times. Each round writes its own run dir `news_r{NN}` (so no round
overwrites another — the LiveRunner truncates *within* a run_id). One round failing (a rate-limit, a
network blip) is caught and logged as a gap — the loop carries on.

In [23]:
import time
from evaluation.runner import run_session

LOG_ROOT = os.path.join(REPO_ROOT, 'experiments', 'runs')
round_runs = []   # [(round_no, run_path-or-None)]

for r in range(1, NUM_ROUNDS + 1):
    config.run_id = f'news_r{r:02d}'
    print(f'\n===== ▶ ROUND {r}/{NUM_ROUNDS}  (run_id={config.run_id}) =====')
    try:
        path = run_session(pipeline, config, game_client=game_client, log_root=LOG_ROOT)
        round_runs.append((r, path))
        print('   round log:', path)
    except Exception as e:
        round_runs.append((r, None))
        print(f'   ⚠️ round {r} FAILED ({type(e).__name__}: {e}) -- logged as a gap, the loop continues.')
    if r < NUM_ROUNDS:
        time.sleep(PAUSE_S)   # polite gap between live games.

print('\nAll rounds done. Logged runs:', [p for _r, p in round_runs if p])


===== ▶ ROUND 1/10  (run_id=news_r01) =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=10450 lvl=0 reached=None -> D | correct=True | timed_out=False | latency=7.4s (left was 29.911534)
[2] qid=11479 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=11.7s (left was 29.909793)
[3] qid=11093 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=11.2s (left was 29.909277)
[4] qid=10862 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=7.1s (left was 29.911035)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[5] qid=11659 lvl=0 reached=None -> D | correct=True | timed_out=False | latency=12.1s (left was 29.910048)
[6] qid=10873 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=7.6s (left was 29.909555)
[7] qid=11119 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=5.9s (left was 29.911565)
[8] qid=11849 lvl=0 reached=7 -> B | correct=False | timed_out=False | latency=11.7s (left was 29.910308)
   round log: /content/NLP/experiments/runs/news_r01

===== ▶ ROUND 2/10  (run_id=news_r02) =====


...

[1] qid=10871 lvl=0 reached=None -> D | correct=True | timed_out=False | latency=8.7s (left was 29.912156)
[2] qid=11870 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=16.0s (left was 29.910801)
[3] qid=12007 lvl=0 reached=2 -> D | correct=False | timed_out=False | latency=8.0s (left was 29.911005)
   round log: /content/NLP/experiments/runs/news_r08

===== ▶ ROUND 9/10  (run_id=news_r09) =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=11802 lvl=0 reached=0 -> B | correct=False | timed_out=False | latency=10.2s (left was 29.911831)
   round log: /content/NLP/experiments/runs/news_r09

===== ▶ ROUND 10/10  (run_id=news_r10) =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=11614 lvl=0 reached=None -> D | correct=True | timed_out=False | latency=8.4s (left was 29.911363)
[2] qid=12022 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=8.4s (left was 29.910866)
[3] qid=11503 lvl=0 reached=2 -> B | correct=False | timed_out=False | latency=11.4s (left was 29.91056)
   round log: /content/NLP/experiments/runs/news_r10

All rounds done. Logged runs: ['/content/NLP/experiments/runs/news_r01', '/content/NLP/experiments/runs/news_r02', '/content/NLP/experiments/runs/news_r03', '/content/NLP/experiments/runs/news_r04', '/content/NLP/experiments/runs/news_r05', '/content/NLP/experiments/runs/news_r06', '/content/NLP/experiments/runs/news_r07', '/content/NLP/experiments/runs/news_r08', '/content/NLP/experiments/runs/news_r09', '/content/NLP/experiments/runs/news_r10']


## 6 · Analysis — per-round scores, retrieval source mix, every wrong question (with evidence)

Aggregates all rounds and **saves** the consolidated records to `experiments/news_test/` so they ride
back to the repo. Three artifacts: a per-round summary, every question (with retrieval source +
snippets), and the wrong questions alone (with the evidence text — the retrieval-miss vs grounding-miss
diagnosis).

In [24]:
import json, collections
from pathlib import Path
import pandas as pd

OUT_DIR = Path(REPO_ROOT) / 'experiments' / 'news_test'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def _read_round(path):
    """One round's records.jsonl -> list of dict rows ([] if missing/empty)."""
    if not path:
        return []
    p = Path(path) / 'records.jsonl'
    if not p.exists():
        return []
    return [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]

def _sources(row):
    """The retrieval SOURCE mix for a row, from retrieved_snippets ('[theguardian.com#..] ' prefix)."""
    out = collections.Counter()
    for s in (row.get('retrieved_snippets') or []):
        if isinstance(s, str) and s.startswith('['):
            out[s[1:].split('#', 1)[0].split(']', 1)[0]] += 1
    return dict(out)

# --- Gather every round ---
summary_rows, all_q, wrong_q = [], [], []
for r, path in round_runs:
    rows = _read_round(path)
    graded = [x for x in rows if x.get('correct') is not None]
    n_correct = sum(1 for x in graded if x.get('correct') is True)
    reached = [x.get('reached_level') for x in rows if x.get('reached_level') is not None]
    summary_rows.append({
        'round': r,
        'answered': len(rows),
        'correct': n_correct,
        'graded': len(graded),
        'accuracy': (n_correct / len(graded)) if graded else float('nan'),
        'reached_level': max(reached) if reached else None,
    })
    for x in rows:
        rec = {'round': r, **x, 'sources': _sources(x)}
        all_q.append(rec)
        if x.get('correct') is False:
            wrong_q.append(rec)

# --- Per-round summary + overall ---
summary = pd.DataFrame(summary_rows)
print('PER-ROUND SUMMARY (News)')
print(summary.to_string(index=False))
tot_c = int(summary['correct'].sum()); tot_g = int(summary['graded'].sum())
lv = [s['reached_level'] for s in summary_rows if s['reached_level'] is not None]
print(f"\nOVERALL: {tot_c}/{tot_g} graded = {tot_c / tot_g:.1%}" if tot_g else '\nOVERALL: no graded answers')
if lv:
    print(f"reached_level over {len(lv)} rounds: min={min(lv)} max={max(lv)} mean={sum(lv)/len(lv):.1f} | {sorted(lv, reverse=True)}")

# --- Retrieval source mix across ALL questions (did the gate route to the browser? did docs land?) ---
src_total = collections.Counter()
for x in all_q:
    src_total.update(x['sources'])
print('\nRETRIEVAL SOURCE MIX (doc count across all', len(all_q), 'questions):', dict(src_total))
fired = sum(1 for x in all_q if x.get('retrieval_used'))
print(f"retrieval fired on {fired}/{len(all_q)} questions")

PER-ROUND SUMMARY (News)
 round  answered  correct  graded  accuracy  reached_level
     1         8        7       8  0.875000              7
     2         8        7       8  0.875000              7
     3        13       12      13  0.923077             12
     4         3        2       3  0.666667              2
     5         6        5       6  0.833333              5
     6         9        8       9  0.888889              8
     7        12       11      12  0.916667             11
     8         3        2       3  0.666667              2
     9         1        0       1  0.000000              0
    10         3        2       3  0.666667              2

OVERALL: 56/66 graded = 84.8%
reached_level over 10 rounds: min=0 max=12 mean=5.6 | [12, 11, 8, 7, 7, 5, 2, 2, 2, 0]

RETRIEVAL SOURCE MIX (doc count across all 66 questions): {'theguardian.com': 91, 'google_news_rss': 142, 'headless_chromium': 37}
retrieval fired on 66/66 questions


In [25]:
# --- Every WRONG question, with the retrieved EVIDENCE (the retrieval-miss vs grounding-miss check) ---
print(f"{'=' * 78}\nEVERY WRONG QUESTION  ({len(wrong_q)} across {NUM_ROUNDS} rounds)\n{'=' * 78}")
for x in wrong_q:
    opts = x.get('options') or {}
    pick = x.get('predicted_answer')
    print(f"\n[round {x['round']}] qid={x['qid']} reached_level={x.get('reached_level')} "
          f"lat={x.get('latency_s', 0):.1f}s sources={x['sources']}")
    print(f"Q: {x['question_text']}")
    for k, v in opts.items():
        print(f"   {k}. {v}" + ('  <-- our pick (WRONG)' if k == pick else ''))
    snips = x.get('retrieved_snippets') or []
    if snips:
        print('   -- retrieved evidence --')
        for s in snips:
            print(f"      {str(s)[:500]}")
    else:
        print('   (no retrieved evidence logged)')

# --- SAVE the consolidated artifacts (ride back to the repo via experiments/) ---
summary.to_csv(OUT_DIR / 'summary.csv', index=False)
with open(OUT_DIR / 'all_questions.jsonl', 'w', encoding='utf-8') as f:
    for x in all_q:
        f.write(json.dumps(x, ensure_ascii=False) + '\n')
with open(OUT_DIR / 'wrong_questions.jsonl', 'w', encoding='utf-8') as f:
    for x in wrong_q:
        f.write(json.dumps(x, ensure_ascii=False) + '\n')
print(f"\nSaved -> {OUT_DIR}/  (summary.csv, all_questions.jsonl [{len(all_q)}], wrong_questions.jsonl [{len(wrong_q)}])")

EVERY WRONG QUESTION  (10 across 10 rounds)

[round 1] qid=11849 reached_level=7 lat=11.7s sources={'theguardian.com': 3, 'google_news_rss': 3}
Q: According to the 2026-05-14 report, what caused the UK Labour Party's Health Secretary to lose confidence in Prime Minister Keir Starmer?
   A. The Prime Minister's decision to resign
   B. A dispute over healthcare policies  <-- our pick (WRONG)
   C. A scandal involving misuse of public funds
   D. The failure in local and regional elections
   -- retrieved evidence --
      [theguardian.com#guardian:0] There is general agreement that, whatever his problems with domestic politics, Sir Keir Starmer has handled his international diplomatic duties as prime minister with aplomb. He joined with all other European leaders in rejecting the Donald Trump-Benjamin Netanyahu war on Iran. He stood with Canada against Trump’s Anschluss politics of saying it should be joined with the US, and with Denmark against Trump’s attempted … ocratic and socialist